# 20 · Concurrency: threads, processes, async

Pipelines wait a lot — on network, disk, databases. **Concurrency** overlaps
that waiting to go faster. The right tool depends on whether work is **I/O-bound**
(waiting) or **CPU-bound** (computing), and Python's **GIL** shapes the choice.

## The GIL in one paragraph

CPython's **Global Interpreter Lock** lets only one thread execute Python
bytecode at a time. So threads **do not** speed up CPU-bound Python — but they
*do* help I/O-bound work, because a thread releases the GIL while waiting on the
network or disk. For CPU parallelism you use multiple **processes**.

Rule of thumb:
- **I/O-bound** (API calls, DB, files) → **threads** or **asyncio**
- **CPU-bound** (parsing, math, compression) → **multiprocessing**

## Threads for I/O-bound work

Fetching many URLs is mostly waiting. A `ThreadPoolExecutor` runs the waits
concurrently. Here we simulate I/O with `time.sleep` and show the speedup.

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor

def fetch(url):
    time.sleep(0.1)          # pretend network latency
    return f'{url}: 200 OK'

urls = [f'/page/{i}' for i in range(10)]

t0 = time.perf_counter()
serial = [fetch(u) for u in urls]
t1 = time.perf_counter()
with ThreadPoolExecutor(max_workers=10) as pool:
    parallel = list(pool.map(fetch, urls))
t2 = time.perf_counter()

print(f'serial:   {(t1 - t0):.2f}s')
print(f'threaded: {(t2 - t1):.2f}s')
print('same results:', serial == parallel)

## asyncio for high-concurrency I/O

`asyncio` runs thousands of I/O tasks on a single thread using an event loop —
lighter than threads for large fan-out (e.g. many API calls). You `await`
coroutines and launch them together with `asyncio.gather`.

> Notebooks already run an event loop, so we use top-level `await` directly
> (in a plain `.py` script you'd wrap the entry point in `asyncio.run(main())`).

In [ ]:
import asyncio, time

async def fetch_async(url):
    await asyncio.sleep(0.1)     # non-blocking wait
    return f'{url}: 200 OK'

urls = [f'/page/{i}' for i in range(10)]
t0 = time.perf_counter()
results = await asyncio.gather(*(fetch_async(u) for u in urls))
print(f'async gathered {len(results)} in {time.perf_counter() - t0:.2f}s')
print(results[0])

## CPU-bound work → multiprocessing

For heavy computation, spread work across **processes** to use multiple cores
(each process has its own GIL). The code pattern with
`ProcessPoolExecutor` looks like this:

```python
from concurrent.futures import ProcessPoolExecutor

def heavy(n):
    return sum(i * i for i in range(n))   # CPU-bound

if __name__ == '__main__':               # required guard on Windows/macOS
    with ProcessPoolExecutor() as pool:
        results = list(pool.map(heavy, [10_000_00] * 8))
```

> We show this as code rather than running it in the notebook: on Windows/macOS
> the worker processes must **import** the target function (spawn start method),
> which doesn't work for functions defined in a notebook cell. Put such
> functions in a `.py` module and call them from a script — as the capstone
> does.

In [ ]:
# Demonstrate the CPU function itself (single-process) so you see the work:
import time
def heavy(n):
    return sum(i * i for i in range(n))

t0 = time.perf_counter()
res = [heavy(200_000) for _ in range(4)]
print('computed', len(res), 'results in', round(time.perf_counter() - t0, 3), 's')
print('sample:', res[0])

### Recap

The GIL means threads help **I/O-bound** work, not CPU-bound; use
`ThreadPoolExecutor` or `asyncio` for network/disk fan-out (asyncio scales to
huge concurrency on one thread); use `multiprocessing`/`ProcessPoolExecutor` for
CPU parallelism, keeping worker functions in importable modules. Next: logging,
config and CLIs.